In [1]:
import pandas as pd

In [2]:
games = pd.read_csv('../data/features.csv')

In [3]:
games.shape

(12300, 46)

In [4]:
home = games[games['HOME'] == 1]
away = games[games['HOME'] == 0]

In [5]:
home.shape

(6140, 46)

In [6]:
away.shape

(6160, 46)

## One row per game
Split the team rows into home and away, then merge them on GAME_ID so each row holds both teams side by side (_HOME / _AWAY). The 10 neutral-site games have no home row, so the merge drops them. Result: 6140 games.

In [7]:
matchups = pd.merge(home, away, on='GAME_ID', suffixes=('_HOME', '_AWAY'))

In [8]:
matchups.shape

(6140, 91)

In [9]:
features = ['ORTG_L10_HOME', 'DRTG_L10_HOME', 'REB_L10_HOME', 'WIN_L10_HOME', 'REST_DAYS_HOME', 'NET_L10_HOME',
            'ORTG_L10_AWAY', 'DRTG_L10_AWAY', 'REB_L10_AWAY', 'WIN_L10_AWAY', 'REST_DAYS_AWAY', 'NET_L10_AWAY']

## Drop games with missing history
Each team's first 10 games of a season have no 10-game average yet (NaN), and the model can't use NaN. Drop those games from matchups first, then rebuild X and y so they stay lined up.

In [10]:
matchups = matchups.dropna(subset=features)

## Target (y)
WIN_HOME: 1 if the home team won, 0 if it lost. This is what the model predicts.

In [11]:
y = matchups['WIN_HOME']

## Features (X)
The 12 pre-game columns the model learns from: the 6 features for each side (_HOME / _AWAY). Nothing from the game itself, so there's no leakage.

In [12]:
X = matchups[features]

In [13]:
X

,ORTG_L10_HOME,DRTG_L10_HOME,REB_L10_HOME,WIN_L10_HOME,REST_DAYS_HOME,NET_L10_HOME,ORTG_L10_AWAY,DRTG_L10_AWAY,REB_L10_AWAY,WIN_L10_AWAY,REST_DAYS_AWAY,NET_L10_AWAY
4,106.143592,115.501296,44.5,0.2,2.0,-9.357705,106.989696,106.781684,46.0,0.4,2.0,0.208012
5,107.265861,114.938874,44.1,0.2,1.0,-7.673013,97.847442,105.538512,43.6,0.2,2.0,-7.691069
6,109.055313,113.860172,44.4,0.3,2.0,-4.804859,103.901276,100.650043,45.1,0.5,2.0,3.251234
7,111.584134,112.482340,43.5,0.4,3.0,-0.898206,103.767842,107.063794,46.3,0.5,1.0,-3.295953
8,111.073568,111.668382,43.4,0.4,2.0,-0.594814,98.763808,101.358424,51.5,0.5,2.0,-2.594616
...,...,...,...,...,...,...,...,...,...,...,...,...
6135,109.853589,122.507639,38.1,0.0,2.0,-12.654050,111.886456,103.150981,45.1,1.0,3.0,8.735474
6136,107.046397,121.241994,37.2,0.1,2.0,-14.195597,112.209744,113.517211,45.2,0.6,2.0,-1.307468
6137,109.823497,124.924768,38.0,0.1,2.0,-15.101271,107.799997,121.877115,43.9,0.1,2.0,-14.077118
6138,109.403977,125.175397,39.1,0.1,2.0,-15.771419,109.383535,117.844016,45.5,0.2,2.0,-8.460480


In [14]:
X.shape, y.shape

((5362, 12), (5362,))

In [15]:
test_mask = matchups['SEASON_ID_HOME'] == 22025

## Train/test split (by time)
Train on 2021-22 to 2024-25, test on 2025-26. The model never sees a test game while learning, so the test score shows how it does on a season it hasn't seen. A random split would let it learn from later games to predict earlier ones.

In [16]:
X_train = X[~test_mask]
y_train = y[~test_mask]

X_test = X[test_mask]
y_test = y[test_mask]

In [17]:
X_train.shape, X_test.shape

((4291, 12), (1071, 12))